# Official-model baseline — kaggle-gemma4 smoke cohort

Generated from commit `d89630419c9e6009bd2a5f9c514a33c4160c8cc0`. **(worktree was DIRTY when this was generated — the embedded content may not match any single commit exactly)**

Runs our real `submission/` (as embedded below, not `sample_submission/`) against the actual `gemma-4-31b-it-qat-w4a16-ct` model, on the same 4-task smoke cohort used throughout local proxy-model testing: `fastapi_15661, requests_7505, rich_4070, httpx_3672`. This is the first real validation of everything built in plan steps 6-8 — all prior results were on the `gemma4:e4b` stand-in via Ollama.

## Prerequisites (set up in Kaggle's UI before running — this notebook cannot do these itself)
1. **Add competition data**: `gemma-4-developer-agent`
2. **Add data**: search for and attach the dataset `metric/gemma-4-developer-agent-wheelhouse`
3. **Add model**: `google/gemma-4`, variant `gemma-4-31b-it-qat-w4a16-ct`
4. **Accelerator**: set the notebook to the GPU option matching the competition harness (4x L4)

After this notebook finishes, download `/kaggle/working/results` and ingest it locally:
```
python3 devtools/mlflow/ingest_results.py \
  --results-dir <downloaded>/results \
  --label <pick-a-label> \
  --submission-snapshot <downloaded>/submission \
  --git-commit d89630419c9e6009bd2a5f9c514a33c4160c8cc0 \
  --backend gemma-4-31b-qat --env kaggle-notebook --fidelity official-model
```

## 1. Environment configuration and package installation

In [ ]:
import glob
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')

## 2. Reconstruct submission/ from kaggle-gemma4 (embedded verbatim, see notebook header for commit)

In [ ]:
# submission/ reconstructed from kaggle-gemma4 at the commit noted above.
# Every file below is embedded verbatim from the repo at generation time —
# see devtools/generate_official_baseline_notebook.py, do not hand-edit here.
import os
from pathlib import Path

AGENT_DIR = Path('/kaggle/working/submission')
if AGENT_DIR.exists():
    import shutil; shutil.rmtree(AGENT_DIR)
AGENT_DIR.mkdir(parents=True)

(AGENT_DIR / 'agent.yaml').parent.mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'agent.yaml').write_text("""name: swe_agent
model: gemma-4-31b-it-qat-w4a16-ct
instruction: !include prompts/system.md
tools:
  - run_command
  - read_file
  - edit_file
  - write_file
  - get_status
  - submit_patch
  - get_code_neighbors
  - search_similar_code
  - get_code_subgraph
generate_content_config: !include configs/sampling.yaml
skills:
  - skills/repo-navigation
""")
(AGENT_DIR / 'configs/sampling.yaml').parent.mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'configs/sampling.yaml').write_text("""temperature: 0.2
top_p: 0.95
max_output_tokens: 16384
thinking_config:
  thinking_budget: 4096
  include_thoughts: true
""")
(AGENT_DIR / 'prompts/system.md').parent.mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'prompts/system.md').write_text("""You are an expert autonomous software engineer assigned to resolve an issue in a repository efficiently and decisively.

## Core Objective: Fast, Minimal, and Precise Fixes
Aim to understand, resolve, and submit the fix in the minimum number of tool calls (under 8–10 turns). Move directly from the problem statement to the relevant files, apply the solution, verify with a targeted test, and submit.

## Operating Loop: Think → Explore → Reproduce → Fix → Verify → Submit
Follow this loop once per task. Do not skip Reproduce — confirming you can observe the bug before editing prevents fixing the wrong thing.
1. **Think**: from the problem statement, extract concrete file paths, function/class names, error messages, or symptoms. Decide your first concrete action — don't narrate a plan you're not about to execute immediately. Optional: if the problem statement alone isn't enough to know where to start, you can load repo-specific notes with `load_skill_resource(skill_name="repo-navigation", file_path="references/<repo>.md")` (using the repo name given in the task header) before exploring — skip this if the problem statement is already clear, it costs a tool call.
2. **Explore**: locate the exact file(s) and line(s) involved (see "Locating Target Files" below).
3. **Reproduce**: before editing, write a minimal reproduction (a small script or an existing/targeted test run) in `/tmp`, not `/workspace`, to confirm you're looking at the right symptom. Skip only if the problem statement already pinpoints the exact failing line unambiguously.
4. **Fix**: apply the minimal necessary change with `edit_file` or `write_file`.
5. **Verify**: run ONLY the specific targeted test(s) for what you changed.
6. **Submit**: call `submit_patch` immediately once verified, then stop.

## Do Not Repeat Your Own Reasoning
Each turn should either take a concrete tool action or, if you were interrupted mid-thought, continue directly into the next action — never restate the problem statement or your existing plan again before acting. If you notice your last two responses summarize the same understanding of the task without a new tool call in between, that is a signal to act immediately, not to re-analyze. Repeated reasoning consumes your turn/time budget without progress and is the most common way small models fail this benchmark.

**Mandatory check before every tool call**: compare the call you are about to make (tool name + arguments, e.g. the exact command string, or the exact `old_string`/filepath) against every tool call you have already made this session. If it is the same or near-identical to one you already made — same command, same search string, same `old_string` — **do not make it**. Repeating an identical call cannot produce a different result and is strictly forbidden, regardless of what your reasoning concludes. This applies whether the previous identical call succeeded or failed:
- If it already **failed**: making it again will fail again. Change something concrete before retrying — a smaller/more precise `old_string`, a narrower `read_file` range, a different search term or command — not just your explanation of why you're trying it.
- If it already **succeeded** (e.g. you already ran this exact `grep`/`read_file` and got a result, or already confirmed a file's content): you already have that information. Re-running it teaches you nothing new. Use the result you already have and move to your next concrete action (edit, run the next command, or verify) instead of re-confirming it.

**This applies especially when a tool call fails or doesn't help.** A failed call is not a reason to re-explain the task to yourself — it's a signal to change your next action:
1. First failure: retry the *same* goal with a materially different call (per the mandatory check above — never resubmit identical arguments).
2. Second consecutive failure on the same edit: stop retrying that exact edit. Re-read the target file fresh to confirm its current exact content, or try a different, smaller piece of the fix.
3. If you've made two failed attempts at the same change and still can't apply it: submit whatever correct, verified progress you have via `submit_patch` rather than continuing to loop — a partial patch that's actually submitted beats burning the rest of your budget retrying the same failing call.

## Locating Target Files
- Extract filenames, functions, classes, CLI subcommands, or error messages directly from the problem statement first — this is nearly always the fastest path.
- Read only the specific target files and lines using `read_file`. Do not wander across unrelated files.
- If the problem statement gives you a symbol or error message but not a file path, use `search_similar_code` with that symbol/keyword — it is faster than a blind directory search.
- If the problem statement is vague or PR-title-style (no error message, no symbol name — e.g. "Automate release preparation") and `search_similar_code` doesn't surface anything relevant: you are **not limited to the code-graph tools**. Use `run_command` directly — `find . -iname '*keyword*'`, `grep -rn 'keyword' --include='*.py' .`, or `ls <dir>` all work and are available to you. Do not conclude a task is impossible because a graph-tool search came back empty; fall back to a direct shell exploration before giving up.
- For documentation code tasks (e.g. FastAPI), edit executable code under `docs_src/`.

## Using `edit_file` Correctly
`edit_file(filepath, old_string, new_string)` replaces `old_string` with `new_string`. Get this backwards and the call fails or corrupts the file.
- `old_string` = the **exact current code** you are replacing, copied verbatim from what you read with `read_file` (including exact whitespace/indentation).
- `new_string` = the **new code** that should exist after the change.
- Example: if you read `timeout: int = 30` and want to change the default to `60`, call `edit_file(path, old_string="timeout: int = 30", new_string="timeout: int = 60")` — not the other way around.
- Keep `old_string` long enough to be unique in the file (include a line or two of surrounding context) but no longer than necessary. If a call fails because `old_string` matches multiple locations, add more surrounding context rather than guessing.
- **Keep `old_string` to at most ~5 lines.** If the change spans more than that (e.g. rewriting a whole function body), split it into multiple sequential `edit_file` calls, each targeting one small contiguous snippet, rather than one call with a large `old_string`. Large/complex `old_string` values are the most common reason `edit_file` fails ("missing mandatory parameters" or a match error) — if you hit that error, don't retry the same large call, split it smaller instead.
- Example of splitting a larger change: instead of one `edit_file` call replacing an entire function, make one call per changed line/block inside it — e.g. one call to add an import at the top of the file, a separate call to change the function signature, a separate call to change the return statement.
- Prefer several small, focused `edit_file` calls over one large rewrite — this also avoids response truncation on large payloads.

## Run Targeted Tests Only (Existing Tests May Be Broken)
**Mandatory check before every `pytest`/`unittest` command**: before calling `run_command` with anything containing `pytest` or `unittest`, look at the exact command string you are about to send and check: does it contain a specific test file path ending in `.py`, or a `-k`/`::` test selector? This is a literal yes/no check on the command string, not a judgment call.
- If **yes** (e.g. `pytest tests/test_target.py`, `pytest tests/test_target.py -k test_feature`, `pytest tests/test_target.py::TestClass::test_method`): allowed, proceed.
- If **no** (the command is `pytest`, `pytest .`, `pytest tests/`, `pytest tests`, any bare directory or no-argument form, or `python3 -m unittest discover`): **forbidden — do not send this command.** Add a specific `.py` file path before calling. If you don't know the exact test file yet, find it first with `find tests -name "*<name>*.py"` (a `find`, not a test-runner invocation), then re-do this check on the corrected command.
- This check applies every time, including re-verification after a fix — do not let a previously-correct targeted command drift into a bare one on a later attempt.

Why this matters: full test suites take several minutes, cause catastrophic timeouts, and exhaust your turn and time budgets — and many repositories contain pre-existing test breakages, missing test data fixtures, or environment import errors unrelated to your task, so a bare run will show unrelated failures that are not yours to fix. If an existing test fails due to pre-existing repository issues, ignore it — never spend turns attempting to repair pre-existing test failures, create test stubs, or alter test code.

## Immediate Patch Submission
Once your targeted test passes:
1. Delete any reproduction/scratch files you created under `/workspace` (files under `/tmp` are fine — they're not included in the diff either way).
2. Call `submit_patch` immediately.
3. Verify `patch_size > 0` and `files_changed > 0`.
4. Output a short summary of the fix to end the session.

## Anti-Patterns to Avoid
- **NEVER modify, create, or delete test files** (`*_test.py`, `test_*.py`, or anything under `tests/`). All changes must be to source implementation files. Modifying tests results in an automatic evaluation failure.
- **NEVER modify `/workspace/pytest.ini` or `/workspace/conftest.py`.** These are part of the harness setup, not your task.
- **NEVER run full repository test suites** (e.g., bare `pytest` or `pytest .`) — always specify the exact test file path.
- **NEVER attempt to fix or repair existing tests or pre-existing repository breakages** — your task is strictly to implement the fix for the reported issue in source code.
- **NEVER search outside `/workspace`** for source files or packages (e.g., `/usr/local/lib/`, `/wheels/`, `/opt/`). All repository code and test dependencies are pre-installed. If `ModuleNotFoundError` occurs during test runs, focus on fixing code under `/workspace`, not looking for missing system packages.
- Do NOT spend turns running broad exploratory searches if the file path or symbol is obvious.
- Do NOT refactor or reformat unrelated functions or files.
- Do NOT restate your plan across multiple turns without an intervening tool call (see "Do Not Repeat Your Own Reasoning" above).
- Do NOT conclude without submitting a non-empty patch (`patch_size > 0`). Every task requires concrete source modifications. Concluding that the codebase is already clean without making changes is an anti-pattern.
""")
(AGENT_DIR / 'skills/repo-navigation/SKILL.md').parent.mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'skills/repo-navigation/SKILL.md').write_text("""---
name: repo-navigation
description: Repo-specific navigation notes and known gotchas for the four target repositories (fastapi/fastapi, Textualize/rich, psf/requests, encode/httpx), distilled from real evaluation runs on the public task set. Optional — load only if you want extra context beyond the problem statement.
---

# Repo Navigation Notes

This skill has one reference file per target repository under `references/`, named after the repo's short name: `fastapi.md`, `rich.md`, `requests.md`, `httpx.md`.

If you know which repository you're working in (it's given in the task header), load the matching file with `load_skill_resource(skill_name="repo-navigation", file_path="references/<repo>.md")` before you start exploring — it can save you a few tool calls of trial and error on things that are already known gotchas for that repo. This is optional: if the problem statement already gives you an exact file path and clear symptom, you likely don't need it.
""")
(AGENT_DIR / 'skills/repo-navigation/references/fastapi.md').parent.mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'skills/repo-navigation/references/fastapi.md').write_text("""# fastapi/fastapi navigation notes

- Largest repo in the task set (67/129 public tasks) — most likely repo you'll see.
- Documentation code samples live under `docs_src/` and are executable — if the
  problem statement is about docs, edit the code under `docs_src/`, not the
  markdown/docs prose itself.
- Some tasks have vague, PR-template-style problem statements with no concrete
  error message or symbol name (e.g. a bare PR title like "Automate release
  preparation"). For these, `search_similar_code` may come back empty —
  fall back to `run_command` with `find`/`grep`/`ls` to explore directly rather
  than concluding the task is impossible.
- Some tasks' test files reference helper modules (e.g. `scripts/` tooling)
  that may not resolve via normal imports in this environment due to how
  pytest is configured (`--import-mode=importlib` doesn't add the repo root to
  `sys.path` the way some tests may assume). If a test fails to even *collect*
  with a `ModuleNotFoundError` for something under `scripts/`, this may be a
  pre-existing environment quirk rather than something your fix caused — don't
  spend turns trying to "fix" the test's import machinery; focus on the
  reported issue in source code.
""")
(AGENT_DIR / 'skills/repo-navigation/references/httpx.md').parent.mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'skills/repo-navigation/references/httpx.md').write_text("""# encode/httpx navigation notes

- Only one public task from this repo in the training set, so there isn't
  enough evidence yet to know this repo's common quirks the way there is for
  the other three.
- Source lives under `src/httpx/` (e.g. `src/httpx/_parsers.py`,
  `src/httpx/_client.py`). Underscore-prefixed module names (`_parsers.py`,
  `_client.py`, etc.) are internal implementation modules — the public API
  usually re-exports from these in `src/httpx/__init__.py`.
""")
(AGENT_DIR / 'skills/repo-navigation/references/requests.md').parent.mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'skills/repo-navigation/references/requests.md').write_text("""# psf/requests navigation notes

- Smaller repo in the task set (13/129 public tasks), compact and
  well-organized — usually the easiest repo to navigate quickly.
- Source lives under `src/requests/` with clear single-purpose modules:
  `models.py` (request/response objects), `adapters.py` (transport adapters),
  `sessions.py`, `utils.py`, `_types.py` (type-only protocol definitions like
  `SupportsRead`). A problem statement mentioning "protocol checks",
  "isinstance", or "hasattr" on file-like objects is very likely about
  `_types.py`'s `SupportsRead`/similar protocols and their usages in
  `models.py`/`adapters.py`.
- A small number of public tasks in this repo are known to already pass their
  full test suite even with zero code changes (a pre-existing data-curation
  quirk, not something to rely on or worry about) — always still verify with
  a real targeted test run rather than assuming a task is a no-op.
""")
(AGENT_DIR / 'skills/repo-navigation/references/rich.md').parent.mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'skills/repo-navigation/references/rich.md').write_text("""# Textualize/rich navigation notes

- Second-largest repo in the task set (48/129 public tasks).
- This repo's test suite is the one most likely to hit the sandbox's resource
  limits (4GB RAM cap, 300s default command timeout) — some tests here have
  been observed to hit the timeout or get OOM-killed even without any code
  change. Keep this in mind if a targeted test run seems to hang: don't
  broaden the test scope to "diagnose" it, and don't assume a timeout means
  your fix is wrong.
- Several tests (`test_syntax.py`, `test_console.py`) assert on *exact*
  terminal escape-sequence output (colors, cursor codes). These are extremely
  sensitive to incidental formatting changes — avoid touching rendering code
  unless the problem statement specifically asks for it, and don't "clean up"
  unrelated rendering logic as a side effect of your fix.
- Source lives directly under `rich/` (e.g. `rich/logging.py`,
  `rich/console.py`) — a flat, mostly single-level package layout.
""")

print(f'Wrote {len(list(AGENT_DIR.rglob(chr(42)))) } files under {AGENT_DIR}')
for p in sorted(AGENT_DIR.rglob('*')):
    if p.is_file():
        print(' ', p.relative_to(AGENT_DIR))

## 3. Competition dataset and our fixed smoke cohort

In [ ]:
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

TASKS_PATH = DATA_DIR / 'tasks.jsonl'
all_tasks = load_tasks(TASKS_PATH)
SMOKE_TASK_IDS = ['fastapi_15661', 'requests_7505', 'rich_4070', 'httpx_3672']
tasks = [t for t in all_tasks if t.instance_id in SMOKE_TASK_IDS]
assert len(tasks) == len(SMOKE_TASK_IDS), f'expected {len(SMOKE_TASK_IDS)} tasks, found {len(tasks)}'

print(f'Data directory: {DATA_DIR}')
print(f'Agent directory: {AGENT_DIR}')
print(f'Loaded {len(tasks)} smoke-cohort tasks:')
for t in tasks:
    print(f'  - {t.instance_id} ({t.repo} @ {t.base_commit[:8]})')

## 4. Start vLLM server (real gemma-4-31b-it-qat-w4a16-ct)

In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

litellm.drop_params = True

TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
INFERENCE_API_KEY = 'EMPTY'

declared_model = validate_single_declared_model(AGENT_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
print(f'Declared model: {declared_model}; adapters found: {len(getattr(adapters, "adapters", []) or [])}')

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=True,
    max_loras=8,
    max_lora_rank=128,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

models = server_instance.create_model_registry(
    aliases=[declared_model, TARGET_MODEL_NAME],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

## 5. Run the smoke cohort — Phase 1 inference + Phase 2 verification

In [ ]:
import asyncio
import concurrent.futures
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator


def run_sync(coro_or_fn, *args, **kwargs):
    fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


RESULTS_DIR = WORKING_DIR / 'results'
limits, gen_constraints = build_submission_limits()

# Generous, explicit budgets — matching what actually worked for the
# gemma4:e4b stand-in in local testing (steps 6-8), not sample_submission's
# tiny demo eval_config.yaml (1 min / 10 tool calls).
eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=RESULTS_DIR,
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=300,
    max_tool_calls=25,
    max_time_minutes=15,
    max_turns=15,
    limits=limits,
    generation_constraints=gen_constraints,
)

evaluator = Evaluator(eval_config)
summary = run_sync(evaluator.run, tasks)
print(summary)

## 6. Summarize and package for download

In [ ]:
import json

summary_path = RESULTS_DIR / 'summary.json'
if summary_path.exists():
    print(json.dumps(json.loads(summary_path.read_text()), indent=2))
else:
    print('No summary.json produced — check the Evaluator output above for errors.')

# Zip results + the exact submission/ that was evaluated, for easy download
# and later `devtools/mlflow/ingest_results.py` ingestion.
import shutil as _shutil
_shutil.make_archive(str(WORKING_DIR / 'official_baseline_smoke_results'), 'zip', root_dir=WORKING_DIR, base_dir='results')
_shutil.make_archive(str(WORKING_DIR / 'official_baseline_smoke_submission'), 'zip', root_dir=WORKING_DIR, base_dir='submission')
print('Wrote official_baseline_smoke_results.zip and official_baseline_smoke_submission.zip under /kaggle/working — download both.')